In [ ]:
%cd ../src

In [ ]:
import logging
from datetime import datetime

import polars as pl
from dateutil.relativedelta import relativedelta

from brokers.kite import KiteLogin
from compute import gen_nse_base_population, run_scanner
from conf import KITE_CREDENTIALS_PATH
from config.brokers import KiteConfig
from config.compute import ScannerConfig
from config.file_paths import FilePathConfig
from ingest import fetch_nse_fno_stocks, fetch_nse_hist_data
from utils import setup_logger

logger = logging.getLogger()
setup_logger()

In [ ]:
END_DATE = datetime(2026, 11, 20, 0, 0, 0)
START_DATE = END_DATE - relativedelta(months=10)
end_date_str = END_DATE.strftime("%Y-%m-%d")
file_paths = FilePathConfig(end_date_str)

In [ ]:
END_DATE

In [ ]:
START_DATE

# Kite Login

In [ ]:
kite_conn = KiteLogin(credentials_path=KITE_CREDENTIALS_PATH).auto_login()

# Historical Data Download

In [ ]:
fetch_nse_hist_data(
    kite_conn=kite_conn,
    start_date=START_DATE.strftime("%Y-%m-%d %H:%M:%S"),
    end_date=END_DATE.strftime("%Y-%m-%d %H:%M:%S"),
    frequency="day",
    save_path=end_date_str,
)

In [ ]:
nse_fno_stock_data = fetch_nse_fno_stocks(kite_conn=kite_conn)

# Load Data

In [ ]:
_tbl_name = KiteConfig.get_db_tbl(
    data_type=KiteConfig.TYP_STOCKS, frequency="day", failed_tbl=False
)

stock_ohlcv_data = pl.read_database_uri(
    query=f"SELECT * from {_tbl_name}", uri=KiteConfig.DB_CONN, engine="connectorx"
)

In [ ]:
base_population_data = gen_nse_base_population(data=stock_ohlcv_data)
base_population_data.write_csv(file_paths.base_population)

In [ ]:
long_data, short_data = run_scanner(
    base_pop_data=base_population_data,
    stock_ohlcv_data=stock_ohlcv_data,
    fno_stocks=nse_fno_stock_data,
    end_date=END_DATE,
    scan_conf=ScannerConfig,
)

In [ ]:
long_data.write_csv(file_paths.long_stocks)
short_data.write_csv(file_paths.short_stocks)